In [1]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425345 sha256=c4e9e72d0d06a07ae15221b11fd535c063c5df22ccbf6b59cc2506eaefafc655
  Stored in directory: /root/.cache/pip/wheels/41/4e/10/c2cf2467f71c678cfc8a6b9ac9241e5e44a01940da8fbb17fc
Successfully built pyspark


In [2]:
from pyspark.sql import SparkSession
session= SparkSession.builder.appName("spark1").getOrCreate()

In [5]:
doctordata=session.read.json("Doctor.json")

In [6]:
print("Schema of dataset:\n",doctordata.schema)
print("Information of columns:",doctordata.columns)
print("Number of records:",doctordata.count())
print("First 2 records:\n",doctordata.show(2))

Schema of dataset:
 StructType([StructField('Age', LongType(), True), StructField('City', StringType(), True), StructField('Name', StringType(), True), StructField('Salary', LongType(), True)])
Information of columns: ['Age', 'City', 'Name', 'Salary']
Number of records: 5
+---+-------------+------+------+
|Age|         City|  Name|Salary|
+---+-------------+------+------+
| 40|          NYC|Wilson|    55|
| 30|Washington Dc| David|    52|
+---+-------------+------+------+
only showing top 2 rows

First 2 records:
 None


In [7]:
#Create a folder named DocDir

In [16]:
stream_json=session.readStream.schema(doctordata.schema).json("DocDir/")

In [17]:
#Copy the json file to the DoctorDir folder
import shutil
src=r"Doctor.json"
dest = r"DocDir"
shutil.copy(src,dest)

'DocDir/Doctor.json'

In [21]:
stream_json.writeStream.queryName("Doctortable").format("memory").outputMode("append").start()


In [23]:
import time
for i in range(10):
    session.sql("select sum(Salary), City from Doctortable group by City").show()
    newfile="DocDir/Doctor" + str(i) +  ".json"
    shutil.copy(src,newfile)
    time.sleep(6)

+-----------+-------------+
|sum(Salary)|         City|
+-----------+-------------+
|         97|Washington DC|
|        145|          NYC|
+-----------+-------------+

+-----------+-------------+
|sum(Salary)|         City|
+-----------+-------------+
|        194|Washington DC|
|        290|          NYC|
+-----------+-------------+

+-----------+-------------+
|sum(Salary)|         City|
+-----------+-------------+
|        291|Washington DC|
|        435|          NYC|
+-----------+-------------+

+-----------+-------------+
|sum(Salary)|         City|
+-----------+-------------+
|        388|Washington DC|
|        580|          NYC|
+-----------+-------------+

+-----------+-------------+
|sum(Salary)|         City|
+-----------+-------------+
|        485|Washington DC|
|        725|          NYC|
+-----------+-------------+

+-----------+-------------+
|sum(Salary)|         City|
+-----------+-------------+
|        582|Washington DC|
|        870|          NYC|
+-----------+--

In [24]:
#Practical Exercise: Display the number of doctors from each city with continuous streaming of data